In [0]:
# Setup paths and read Silver accounts

from pyspark.sql.functions import *
from pyspark.sql.types import *

accounts_path = "/Volumes/workspace/default/banking-transactions-lakehouse-project-selected-dataset-edition/delta/silver/accounts"
customers_path = "/Volumes/workspace/default/banking-transactions-lakehouse-project-selected-dataset-edition/delta/silver/customers"

df_accounts = spark.read.format("delta").load(accounts_path)
print(f"Total accounts: {df_accounts.count():,}")
df_accounts.printSchema()
df_accounts.show(10, truncate=False)

Total accounts: 10
root
 |-- account_id: string (nullable = true)
 |-- latest_balance: double (nullable = true)
 |-- account_type: string (nullable = true)
 |-- branch_id: string (nullable = true)

+-------------+----------------+------------+---------+
|account_id   |latest_balance  |account_type|branch_id|
+-------------+----------------+------------+---------+
|1196428'     |-1.68284755405E9|Savings     |B4       |
|1196711'     |-1.58691558925E9|Savings     |B1       |
|409000362497'|-1.90190209261E9|Savings     |B1       |
|409000405747'|-5.4826745865E8 |Savings     |B2       |
|409000425051'|-3.5673478865E8 |Savings     |B3       |
|409000438611'|-5.4821049986E8 |Savings     |B1       |
|409000438620'|-5.3945395387E8 |Savings     |B3       |
|409000493201'|743583.32       |Savings     |B4       |
|409000493210'|-5.4624972234E8 |Savings     |B2       |
|409000611074'|462200.0        |Savings     |B1       |
+-------------+----------------+------------+---------+



In [0]:
# Create customer_id from account_id

df_customers = df_accounts.withColumn("customer_id", col("account_id"))

df_customers.select("customer_id", "account_id").show(10, truncate=False)

+-------------+-------------+
|customer_id  |account_id   |
+-------------+-------------+
|1196428'     |1196428'     |
|1196711'     |1196711'     |
|409000362497'|409000362497'|
|409000405747'|409000405747'|
|409000425051'|409000425051'|
|409000438611'|409000438611'|
|409000438620'|409000438620'|
|409000493201'|409000493201'|
|409000493210'|409000493210'|
|409000611074'|409000611074'|
+-------------+-------------+



In [0]:
# Create customer_name column

df_customers = df_customers.withColumn("customer_name", concat(lit("Customer_"), col("account_id")))

df_customers.select("customer_id", "account_id", "customer_name").show(10, truncate=False)

+-------------+-------------+----------------------+
|customer_id  |account_id   |customer_name         |
+-------------+-------------+----------------------+
|1196428'     |1196428'     |Customer_1196428'     |
|1196711'     |1196711'     |Customer_1196711'     |
|409000362497'|409000362497'|Customer_409000362497'|
|409000405747'|409000405747'|Customer_409000405747'|
|409000425051'|409000425051'|Customer_409000425051'|
|409000438611'|409000438611'|Customer_409000438611'|
|409000438620'|409000438620'|Customer_409000438620'|
|409000493201'|409000493201'|Customer_409000493201'|
|409000493210'|409000493210'|Customer_409000493210'|
|409000611074'|409000611074'|Customer_409000611074'|
+-------------+-------------+----------------------+



In [0]:
# Add city column with random values

cities = ["Hyderabad", "Mumbai", "Delhi", "Chennai", "Bangalore"]

# Use hash function to randomly assign cities (since account_id is string)
df_customers = df_customers.withColumn(
    "city",
    when((hash(col("account_id")) % 5) == 0, lit("Hyderabad"))
    .when((hash(col("account_id")) % 5) == 1, lit("Mumbai"))
    .when((hash(col("account_id")) % 5) == 2, lit("Delhi"))
    .when((hash(col("account_id")) % 5) == 3, lit("Chennai"))
    .otherwise(lit("Bangalore"))
)

print("City distribution:")
df_customers.groupBy("city").count().orderBy("city").show()

City distribution:
+---------+-----+
|     city|count|
+---------+-----+
|Bangalore|    6|
|    Delhi|    1|
|Hyderabad|    2|
|   Mumbai|    1|
+---------+-----+



In [0]:
# Ensure no duplicates

print(f"Before deduplication: {df_customers.count():,}")
df_customers = df_customers.dropDuplicates(["customer_id"])
print(f"After deduplication: {df_customers.count():,}")

Before deduplication: 10
After deduplication: 10


In [0]:
# Select final columns

df_customers = df_customers.select(
    "customer_id",
    "account_id",
    "customer_name",
    "city"
)

df_customers.printSchema()
df_customers.show(10, truncate=False)

root
 |-- customer_id: string (nullable = true)
 |-- account_id: string (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- city: string (nullable = false)

+-------------+-------------+----------------------+---------+
|customer_id  |account_id   |customer_name         |city     |
+-------------+-------------+----------------------+---------+
|1196711'     |1196711'     |Customer_1196711'     |Bangalore|
|409000362497'|409000362497'|Customer_409000362497'|Bangalore|
|409000438611'|409000438611'|Customer_409000438611'|Bangalore|
|409000425051'|409000425051'|Customer_409000425051'|Delhi    |
|409000405747'|409000405747'|Customer_409000405747'|Hyderabad|
|1196428'     |1196428'     |Customer_1196428'     |Bangalore|
|409000611074'|409000611074'|Customer_409000611074'|Bangalore|
|409000493201'|409000493201'|Customer_409000493201'|Mumbai   |
|409000438620'|409000438620'|Customer_409000438620'|Bangalore|
|409000493210'|409000493210'|Customer_409000493210'|Hyderabad|
+-------

In [0]:
# Save as Delta table

df_customers.write.format("delta").mode("overwrite").save(customers_path)
print(f"✓ Saved to: {customers_path}")

✓ Saved to: /Volumes/workspace/default/banking-transactions-lakehouse-project-selected-dataset-edition/delta/silver/customers


In [0]:
# Display sample data and summary

print("Final Customer Table:")
df_customers.orderBy("customer_id").show(20, truncate=False)

print("\nSummary:")
print(f"Total customers: {df_customers.count():,}")
print(f"\nCustomers per city:")
df_customers.groupBy("city").count().orderBy("city").show()

Final Customer Table:
+-------------+-------------+----------------------+---------+
|customer_id  |account_id   |customer_name         |city     |
+-------------+-------------+----------------------+---------+
|1196428'     |1196428'     |Customer_1196428'     |Bangalore|
|1196711'     |1196711'     |Customer_1196711'     |Bangalore|
|409000362497'|409000362497'|Customer_409000362497'|Bangalore|
|409000405747'|409000405747'|Customer_409000405747'|Hyderabad|
|409000425051'|409000425051'|Customer_409000425051'|Delhi    |
|409000438611'|409000438611'|Customer_409000438611'|Bangalore|
|409000438620'|409000438620'|Customer_409000438620'|Bangalore|
|409000493201'|409000493201'|Customer_409000493201'|Mumbai   |
|409000493210'|409000493210'|Customer_409000493210'|Hyderabad|
|409000611074'|409000611074'|Customer_409000611074'|Bangalore|
+-------------+-------------+----------------------+---------+


Summary:
Total customers: 10

Customers per city:
+---------+-----+
|     city|count|
+-------